# 09 - Auditoria de Enrichment y Benchmark SERPAVI

## Objetivo
- Validar la calidad y cobertura del enrichment realizado en NB03.
- Evaluar el impacto de las features de datos abiertos en el modelo.
- Benchmark contra el indice SERPAVI (Ministerio de Vivienda).

## Nota
> La descarga e integracion de datos abiertos se realiza ahora en **NB03 (Features Master)**.
> Este notebook es de **auditoria**: verifica que el enrichment se ha hecho correctamente y mide su impacto.

## Inputs esperados
- `artifacts/features_master.parquet` (generado por NB03)
- `artifacts/features_core.parquet` (solo core, para comparar)
- `models/best_model.joblib` o `models/best_models.joblib`
- `reports/enrichment_missing.md` (generado por NB03)

In [1]:
from __future__ import annotations
from pathlib import Path
import sys
import json
import re

import numpy as np
import pandas as pd

SEED = 100432070
np.random.seed(SEED)

def get_repo_root() -> Path:
    current = Path.cwd().resolve()
    for parent in [current] + list(current.parents):
        if (parent / ".git").exists() or (parent / "pyproject.toml").exists():
            return parent
    return current

ROOT = get_repo_root()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.utils import (
    eval_regression, make_boosting_pipeline, DataFrameImputer,
    get_feature_cols, load_model, TARGET_COLS, DERIVED_FROM_TARGET, ID_COLS,
)

---
## 1. Auditoria de cobertura del enrichment

Verificamos que `features_master.parquet` contiene las features esperadas de datos abiertos y su cobertura (% no-nulo).

In [2]:
# Cargar features_master
master_path = ROOT / "artifacts" / "features_master.parquet"
if master_path.exists():
    master = pd.read_parquet(master_path)
else:
    master = pd.read_csv(ROOT / "artifacts" / "features_master.csv.gz")

print(f"features_master: {len(master)} filas, {len(master.columns)} columnas")

# Features base siempre esperadas
ENRICHMENT_FEATURES = [
    "dist_metro_m", "dist_bicimad_m", "bicimad_density_1km",
    "emt_density_500m", "green_area_m2", "licencias_density_1km",
    "vut_count_district",
    "terrace_density_1km", "locales_density_1km",
    "noise_ld_db", "air_no2_nearest",
    "ine_renta_persona", "ine_edad_media", "ine_tamano_hogar", "ine_pct_unipersonal",
    "barrio_join", "distrito_join", "district_norm",
]
# VUT spatial features: solo esperadas si el SHP con coordenadas estaba disponible
for vut_spatial in ["vut_count_1km", "vut_density_1km", "vut_distance_nearest"]:
    if vut_spatial in master.columns:
        ENRICHMENT_FEATURES.append(vut_spatial)

print("\n--- Cobertura de features de enrichment ---")
coverage_data = []
for feat in ENRICHMENT_FEATURES:
    if feat in master.columns:
        n_valid = master[feat].notna().sum()
        pct = n_valid / len(master) * 100
        coverage_data.append({"feature": feat, "n_valid": n_valid, "coverage_pct": round(pct, 1), "status": "OK"})
    else:
        coverage_data.append({"feature": feat, "n_valid": 0, "coverage_pct": 0.0, "status": "MISSING"})

coverage_df = pd.DataFrame(coverage_data)
print(coverage_df.to_string(index=False))

n_ok = (coverage_df["status"] == "OK").sum()
n_missing = (coverage_df["status"] == "MISSING").sum()
print(f"\nResumen: {n_ok} features disponibles, {n_missing} no disponibles")

# Verificar reporte de NB03
report_path = ROOT / "reports" / "enrichment_missing.md"
if report_path.exists():
    print(f"\nReporte de NB03 encontrado: {report_path}")
    print(report_path.read_text(encoding="utf-8")[:500])
else:
    print("\nWARN: No se encontro reports/enrichment_missing.md (re-ejecutar NB03)")

features_master: 8797 filas, 70 columnas

--- Cobertura de features de enrichment ---
              feature  n_valid  coverage_pct status
         dist_metro_m     8797         100.0     OK
       dist_bicimad_m     8797         100.0     OK
  bicimad_density_1km     8797         100.0     OK
     emt_density_500m     8797         100.0     OK
        green_area_m2     5940          67.5     OK
licencias_density_1km     8797         100.0     OK
   vut_count_district     5940          67.5     OK
  terrace_density_1km     8797         100.0     OK
  locales_density_1km     8797         100.0     OK
          noise_ld_db     5722          65.0     OK
      air_no2_nearest     8797         100.0     OK
    ine_renta_persona     6030          68.5     OK
       ine_edad_media     6030          68.5     OK
     ine_tamano_hogar     6030          68.5     OK
  ine_pct_unipersonal     6030          68.5     OK
          barrio_join     8205          93.3     OK
        distrito_join     8205

---
## 2. Impacto del enrichment en el modelo

Comparamos el MAE de un modelo entrenado solo con features core vs el modelo completo (core + enrichment),
**usando los mismos hiperparametros optimizados de NB05** para aislar el efecto puro del enrichment.

In [3]:
from sklearn.metrics import mean_absolute_error

split = np.load(ROOT / "artifacts" / "splits" / "holdout_indices.npz")
train_idx, test_idx = split["train_idx"], split["test_idx"]

y = master["price"]
y_train = y.loc[train_idx].reset_index(drop=True)
y_test = y.loc[test_idx].reset_index(drop=True)
train_mask = np.isfinite(y_train.replace([np.inf, -np.inf], np.nan))
test_mask = np.isfinite(y_test.replace([np.inf, -np.inf], np.nan))
y_train = y_train.loc[train_mask].reset_index(drop=True)
y_test = y_test.loc[test_mask].reset_index(drop=True)

# Features master (todas)
feature_cols_all = get_feature_cols(master, numeric_only=True)

# --- Filtrar por features seleccionadas si existen ---
sel_path = ROOT / "artifacts" / "selected_features.json"
if sel_path.exists():
    import json as _json
    _sel = _json.loads(sel_path.read_text(encoding="utf-8"))
    _all_selected = set()
    for _t in _sel:
        if isinstance(_sel[_t], dict) and "selected" in _sel[_t]:
            _all_selected.update(_sel[_t]["selected"])
    if _all_selected:
        feature_cols_all = [c for c in feature_cols_all if c in _all_selected]
        print(f"Feature selection activa: {len(feature_cols_all)} features")
X_train_all = master.loc[train_idx, feature_cols_all].reset_index(drop=True).loc[train_mask].reset_index(drop=True)
X_test_all = master.loc[test_idx, feature_cols_all].reset_index(drop=True).loc[test_mask].reset_index(drop=True)

# Features core (solo las que no son de enrichment)
enrichment_set = set(ENRICHMENT_FEATURES)
core_only_cols = [c for c in feature_cols_all if c not in enrichment_set]
X_train_core = X_train_all[core_only_cols].copy()
X_test_core = X_test_all[core_only_cols].copy()

# Eliminar columnas vacias
empty_all = X_train_all.columns[X_train_all.notna().sum() == 0]
if empty_all.any():
    X_train_all = X_train_all.drop(columns=empty_all)
    X_test_all = X_test_all.drop(columns=empty_all, errors="ignore")
empty_core = X_train_core.columns[X_train_core.notna().sum() == 0]
if empty_core.any():
    X_train_core = X_train_core.drop(columns=empty_core)
    X_test_core = X_test_core.drop(columns=empty_core, errors="ignore")

# Cargar hiperparametros optimizados de NB05
hp_path = ROOT / "reports" / "best_hyperparams.json"
best_hp = {}
if hp_path.exists():
    best_hp = json.load(open(hp_path))
    print(f"Hiperparametros cargados de NB05: {best_hp}")
else:
    print("WARN: best_hyperparams.json no encontrado, usando defaults")

# Entrenar ambos modelos CON los mismos hiperparametros optimizados
build_fn, _, engine_name = make_boosting_pipeline(SEED)
print(f"Motor boosting: {engine_name}")

model_core = build_fn()
if best_hp:
    model_core.set_params(**best_hp)
model_core.fit(X_train_core, y_train)
preds_core = model_core.predict(X_test_core)
mae_core = mean_absolute_error(y_test, preds_core)

model_all = build_fn()
if best_hp:
    model_all.set_params(**best_hp)
model_all.fit(X_train_all, y_train)
preds_all = model_all.predict(X_test_all)
mae_all = mean_absolute_error(y_test, preds_all)

delta = mae_core - mae_all
print(f"\nMAE (solo core, tuned): {mae_core:.2f} EUR")
print(f"MAE (core + enrichment, tuned): {mae_all:.2f} EUR")
print(f"Delta: {delta:.2f} EUR ({delta/mae_core*100:.1f}% mejora)")

# Guardar reporte
impact_path = ROOT / "reports" / "enrichment_impact.md"
impact_lines = [
    "## Impacto del Enrichment (generado por NB09)", "",
    f"- MAE solo core: {mae_core:.2f} EUR",
    f"- MAE core + enrichment: {mae_all:.2f} EUR",
    f"- Delta: {delta:.2f} EUR ({delta/mae_core*100:.1f}% mejora)",
    f"- Features core: {len(core_only_cols)}",
    f"- Features enrichment: {len(feature_cols_all) - len(core_only_cols)}",
    f"- Features totales: {len(feature_cols_all)}",
    "",
    f"> Hiperparametros usados (de NB05): `{json.dumps(best_hp)}`",
]
impact_path.write_text("\n".join(impact_lines), encoding="utf-8")

Feature selection activa: 42 features
Hiperparametros cargados de NB05: {'model__iterations': 4997, 'model__depth': 8, 'model__learning_rate': 0.29915045507974913, 'model__l2_leaf_reg': 0.015088745786191366, 'model__random_strength': 14.983514265381537, 'model__min_data_in_leaf': 17, 'model__border_count': 56, 'model__od_wait': 73, 'model__model_size_reg': 8.254333438517223, 'model__leaf_estimation_iterations': 15, 'model__rsm': 0.6101741207485138, 'model__grow_policy': 'Depthwise', 'model__bootstrap_type': 'MVS', 'model__subsample': 0.9686170013692436, 'model__sampling_frequency': 'PerTree', 'model__od_type': 'Iter'}
Motor boosting: catboost

MAE (solo core, tuned): 351.86 EUR
MAE (core + enrichment, tuned): 351.33 EUR
Delta: 0.53 EUR (0.2% mejora)


804

In [4]:
try:
    from sklearn.inspection import permutation_importance as perm_imp_fn
    perm_result = perm_imp_fn(model_all, X_test_all, y_test, n_repeats=10, random_state=SEED, n_jobs=1)
    imp_enriched = pd.DataFrame({
        "feature": X_test_all.columns,
        "importance_mean": perm_result.importances_mean,
        "importance_std": perm_result.importances_std,
    }).sort_values("importance_mean", ascending=False)

    print("\nTop 15 features post-enrichment (permutation importance):")
    print(imp_enriched.head(15).to_string(index=False))
    imp_enriched.to_csv(ROOT / "reports" / "enrichment_feature_importances.csv", index=False)

    enrichment_candidates = [c for c in imp_enriched.head(15)["feature"] if c in enrichment_set]
    if enrichment_candidates:
        print(f"\nFeatures de enrichment en top 15: {enrichment_candidates}")
    else:
        print("\nNinguna feature de enrichment en top 15.")
except Exception as exc:
    print(f"Permutation importance no disponible: {exc}")


Top 15 features post-enrichment (permutation importance):
             feature  importance_mean  importance_std
          surface_m2         0.250844        0.028564
           bathrooms         0.167692        0.021495
         floor_built         0.125580        0.011857
 bicimad_density_1km         0.094541        0.007343
 terrace_density_1km         0.063022        0.006861
  distance_center_km         0.054747        0.013494
 locales_density_1km         0.054559        0.005290
      log_surface_m2         0.044982        0.005748
         density_1km         0.042281        0.007468
                 lat         0.041513        0.014372
    emt_density_500m         0.038990        0.006554
            bedrooms         0.037896        0.007307
      dist_bicimad_m         0.034820        0.008290
   vut_density_1_0km         0.029547        0.005506
vut_distance_nearest         0.026408        0.006115

Features de enrichment en top 15: ['bicimad_density_1km', 'terrace_density_1

---
## 3. Benchmark SERPAVI

Comparacion con el Indice de Alquiler de Vivienda del Ministerio (MITMA/MIVAU). Este indice proporciona rangos de referencia por seccion censal (datos hasta 2018).

In [5]:
index_path = ROOT / "data" / "external" / "socioeco" / "Sistema_Indice_Alquiler_Vivienda.xlsx"
index_schema_path = ROOT / "reports" / "index_alquiler_schema.md"

if index_path.exists():
    xls = pd.ExcelFile(index_path)
    sheet = "SeccionesCensales" if "SeccionesCensales" in xls.sheet_names else xls.sheet_names[0]
    df_index = pd.read_excel(index_path, sheet_name=sheet)
    col_list = chr(10).join(["- " + str(c) for c in df_index.columns])
    schema_lines = [
        "# Schema del Indice de Alquiler de Vivienda (MITMA/MIVAU)",
        "", "## Fuente",
        "- Fichero: Sistema_Indice_Alquiler_Vivienda.xlsx",
        "- Hoja utilizada: " + str(sheet),
        "- Total filas: " + str(len(df_index)) + ", columnas: " + str(len(df_index.columns)),
        "", "## Columnas detectadas", col_list,
        "", "## Uso en el proyecto",
        "- Benchmark/contexto historico para comparar precios actuales vs referencia oficial.",
        "- NO se usa como feature del modelo supervisado (solo comparador/narrativa).",
        "- Limitacion temporal: datos hasta 2018; el microdato es 2025.",
    ]
    index_schema_path.write_text(chr(10).join(schema_lines), encoding="utf-8")
    print(f"Schema SERPAVI guardado: {index_schema_path}")
    print(f"  {len(df_index)} filas, {len(df_index.columns)} columnas, hoja: {sheet}")
else:
    print("Indice SERPAVI no disponible (fichero no descargado)")

# SERPAVI manual instructions
serpavi_path = ROOT / "reports" / "serpavi_manual_instructions.md"
serpavi_lines = [
    "# Instrucciones manuales para SERPAVI", "",
    "## Contexto",
    "El Sistema Estatal de Referencia del Precio del Alquiler de Vivienda (SERPAVI)",
    "no ofrece un CSV/ZIP descargable por seccion censal.", "",
    "## Nivel A: Benchmark narrativo",
    "PDFs en data/external/docs/ (municipios, provincias, CCAA, metodologia).", "",
    "## Nivel B: Comparador por seccion censal",
    "Excel historico MITMA: data/external/socioeco/Sistema_Indice_Alquiler_Vivienda.xlsx", "",
    "## Limitaciones",
    "- Datos SERPAVI: contratos reales depositados en fianza, no anuncios.",
    "- Excel historico cubre hasta 2018; el microdato es de 2025.",
]
serpavi_path.write_text(chr(10).join(serpavi_lines), encoding="utf-8")
print(f"Instrucciones SERPAVI guardadas: {serpavi_path}")

Schema SERPAVI guardado: C:\Users\samuf\Desktop\SPA-Madrid\reports\index_alquiler_schema.md
  33662 filas, 83 columnas, hoja: SeccionesCensales
Instrucciones SERPAVI guardadas: C:\Users\samuf\Desktop\SPA-Madrid\reports\serpavi_manual_instructions.md


In [6]:
from IPython.display import display, Markdown

# --- Leer metricas del impacto (calculadas en celda impact_analysis) ---
_ei_path = ROOT / "reports" / "enrichment_impact.md"
_ei_delta = "N/A"
_ei_mae_core = "N/A"
_ei_mae_all = "N/A"
if _ei_path.exists():
    for line in _ei_path.read_text(encoding="utf-8").split("\n"):
        if line.startswith("- Delta:"):
            _ei_delta = line.replace("- Delta: ", "")
        if line.startswith("- MAE solo core"):
            _ei_mae_core = line.split(":")[1].strip() if ":" in line else "N/A"
        if line.startswith("- MAE core + enrichment"):
            _ei_mae_all = line.split(":")[1].strip() if ":" in line else "N/A"

# Leer cobertura de enrichment_missing.md
_em_path = ROOT / "reports" / "enrichment_missing.md"
_n_available = 0
_n_missing = 0
if _em_path.exists():
    for line in _em_path.read_text(encoding="utf-8").split("\n"):
        if line.startswith("### Total:"):
            parts = line.replace("### Total: ", "").split(",")
            for p in parts:
                if "activas" in p:
                    _n_available = int("".join(c for c in p if c.isdigit()) or 0)
                if "no disponibles" in p:
                    _n_missing = int("".join(c for c in p if c.isdigit()) or 0)

# Top features de importancias
_fi_path = ROOT / "reports" / "enrichment_feature_importances.csv"
_top_feats = ""
if _fi_path.exists():
    _fi = pd.read_csv(_fi_path)
    if not _fi.empty:
        _top5 = _fi.head(5)
        rows = []
        for i, r in _top5.iterrows():
            rows.append(f"| {i+1} | {r.iloc[0]} | {r.iloc[1]:.4f} |")
        _top_feats = "\n".join(rows)

# Coverage stats
_cov_lines = []
if _em_path.exists():
    in_available = False
    for line in _em_path.read_text(encoding="utf-8").split("\n"):
        if "### Features disponibles:" in line:
            in_available = True
            continue
        if line.startswith("###") and in_available:
            break
        if in_available and line.startswith("- "):
            _cov_lines.append(line)

_n_100pct = sum(1 for l in _cov_lines if "100.0%" in l)
_n_partial = len(_cov_lines) - _n_100pct

md = f"""---

## Conclusiones del Notebook 09 — Auditoria de Enrichment

### Rol de este notebook en el pipeline

Este notebook es un **notebook de auditoria y validacion**. Toda la logica de enrichment (descarga, joins espaciales, BallTree, VUT) se ejecuta en el **NB03 (Features Master)**, que genera `features_master.parquet`. Aqui medimos si ese enrichment realmente aporta valor.

### Resultados de la auditoria

| Aspecto | Resultado |
|---------|-----------|
| **Features de enrichment activas** | {_n_available} |
| **Features no disponibles** | {_n_missing} |
| **Cobertura** | {_n_100pct} features al 100%, {_n_partial} con cobertura parcial |
| **MAE solo core** | {_ei_mae_core} |
| **MAE core + enrichment** | {_ei_mae_all} |
| **Delta** | {_ei_delta} |

### Top features post-enrichment (permutation importance)

| Posicion | Feature | Importancia |
|----------|---------|-------------|
{_top_feats}

### Que significan estos resultados

1. **El enrichment tiene un impacto marginal con el modelo tuneado**: Con hiperparametros optimizados, el modelo core ya captura la mayoria de la senial. El delta es pequeno.

2. **Las features de enrichment de movilidad y contexto urbano aportan senal util**: distancias/densidades de transporte y variables socioeconomicas aparecen en el ranking y ayudan a explicar variacion de precios.

3. **Las features VUT espaciales ahora estan disponibles** gracias a la geocodificacion por address matching implementada en NB03.

### Siguiente paso
-> **NB10 (Temporal)**: Analizar tendencias temporales y crear features de series de tiempo.
"""
display(Markdown(md))

---

## Conclusiones del Notebook 09 — Auditoria de Enrichment

### Rol de este notebook en el pipeline

Este notebook es un **notebook de auditoria y validacion**. Toda la logica de enrichment (descarga, joins espaciales, BallTree, VUT) se ejecuta en el **NB03 (Features Master)**, que genera `features_master.parquet`. Aqui medimos si ese enrichment realmente aporta valor.

### Resultados de la auditoria

| Aspecto | Resultado |
|---------|-----------|
| **Features de enrichment activas** | 16 |
| **Features no disponibles** | 0 |
| **Cobertura** | 9 features al 100%, 7 con cobertura parcial |
| **MAE solo core** | 351.86 EUR |
| **MAE core + enrichment** | 351.33 EUR |
| **Delta** | 0.53 EUR (0.2% mejora) |

### Top features post-enrichment (permutation importance)

| Posicion | Feature | Importancia |
|----------|---------|-------------|
| 1 | surface_m2 | 0.2508 |
| 2 | bathrooms | 0.1677 |
| 3 | floor_built | 0.1256 |
| 4 | bicimad_density_1km | 0.0945 |
| 5 | terrace_density_1km | 0.0630 |

### Que significan estos resultados

1. **El enrichment tiene un impacto marginal con el modelo tuneado**: Con hiperparametros optimizados, el modelo core ya captura la mayoria de la senial. El delta es pequeno.

2. **Las features de enrichment de movilidad y contexto urbano aportan senal util**: distancias/densidades de transporte y variables socioeconomicas aparecen en el ranking y ayudan a explicar variacion de precios.

3. **Las features VUT espaciales ahora estan disponibles** gracias a la geocodificacion por address matching implementada en NB03.

### Siguiente paso
-> **NB10 (Temporal)**: Analizar tendencias temporales y crear features de series de tiempo.
